# Домашнє завдання: Інтерактивні візуалізації з Plotly

## Опис завдання
У цьому домашньому завданні ви будете створювати інтерактивні візуалізації з допомогою бібліотеки Plotly. Ви дізнаєтесь різницю між Plotly Express (швидкі графіки) та Graph Objects (повний контроль), та створите інтерактивний дашборд.

**Опис колонок:**
- `datetime` - дата та час
- `season` - квартал (1-Q1, 2-Q2, 3-Q3, 4-Q4)
- `holiday` - чи є день святковим (0=ні, 1=так)
- `workingday` - чи є день робочим (0=ні, 1=так)
- `weather` - погодні умови (1=ясно, 2=туман, 3=легкий дощ, 4=сильний дощ)
- `temp` - температура в градусах Цельсія
- `atemp` - відчувається як температура
- `humidity` - вологість (%)
- `windspeed` - швидкість вітру
- `casual` - кількість випадкових користувачів
- `registered` - кількість зареєстрованих користувачів
- `count` - загальна кількість орендованих велосипедів

## Підготовка даних

---

🌱 Коментар щодо сезонності

Колонка season у датасеті представляє саме квартали року, а не метеорологічні сезони. Тому всі аналізи сезонності ви можете будувати на основі кварталів.

Водночас дані були зібрані в Індії, де поділ на сезони інший, ніж у Європі чи США. Якщо ви хочете дослідити сезонність відповідно до індійської системи сезонів, можна створити окрему колонку.

Справжні сезони в Індії:

| Сезон        | Місяці                     |
| ------------ | -------------------------- |
| Winter       | December–February (12,1,2) |
| Summer       | March–May (3,4,5)          |
| Monsoon      | June–September (6,7,8,9)   |
| Post-monsoon | October–November (10,11)   |


Тоді потрібно зробити нову колонку weather_season_india, мапнувши місяці так:

12, 1, 2 → 1 (Winter)

3, 4, 5 → 2 (Summer)

6–9 → 3 (Monsoon)

10–11 → 4 (Post-Monsoon)


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Завантаження даних
df = pd.read_csv('drive/MyDrive/yulu_bike_sharing_dataset.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

# Для plotly краще не встановлювати datetime як індекс
df['month'] = df['datetime'].dt.month
df['hour'] = df['datetime'].dt.hour
df['weekday'] = df['datetime'].dt.day_name()

# Додаємо назви кварталів в окрему колонку
quarter_map = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}
df['quarter_name'] = df['season'].map(quarter_map)

In [3]:
# ДОдамо колонку weather_season_india
def ind_season(val):
  if val in [12, 1, 2]:
    return 1
  elif val in [3, 4, 5]:
    return 2
  elif val in [6, 7, 8, 9]:
    return 3
  elif val in [10, 11]:
    return 4

df['weather_season_india'] = df['month'].apply(ind_season)

## Завдання 1: Базовий інтерактивний лінійний графік (Plotly Express)

**Завдання:**
Створіть інтерактивний лінійний графік динаміки оренди за часом (рівень деталізації - як в даних) з можливістю zoom та hover.

Дайте відповіді на питання.
**Питання для інтерпретації:**
1. Яка перевага інтерактивного графіка над статичним?
2. Чому на графіку є "пробіли" - ділянки, де одна пряма лінія зʼєднує два "суцільних" блоки з даними? Як би ви це могли дослідити на статичному графіку?


In [4]:
px.line(df, x='datetime', y='count',
        title='Інтерактивний графік динаміки оренди велосипедів')

# 1.Переваги інтерактивного графіку - при наведенні на точку на графіку, з'являються
#   підказки з підписами даних; є можливість зблизити/віддалити зображення, без
#   втрати якості та дослідити конкретні точки даних.
# 2.'Пробіли' - це відсутні дані за період з 20го числа по кінець місяця в цьому датасеті.
#   Використовуючи метод 'resample' при дослідженні часових рядів, на статистичних
#   графіках також помітно відсутні дані за період (якщо я правильно зрозуміла питання).

## Завдання 2: Scatter plot з додатковими даними (Plotly Express)

**Завдання:**
Створіть scatter plot кількості орендованих велосипедів випадковими користувачами vs кількості орендованих велосипедів зареєстрованими користувачами. Розмір точок встановіть за сумарною кількістю велосипедів, які були взяті в оренду, а колір - за сезоном(кварталом). В hover_data - додайте деталі, які допоможуть вам в подальшому аналізі.

Дослідіть графік. Зверніть увагу, що ви можете вмикати і вимикати окремі квартали, якщо будете клікати на колір кварталу в легенді графіку.

**Дайте відповідь на питання.**
- Як ви проінтерпретуєте роздвоєність цього графіку (дві явні лінії)? Що це означає?
- Які висновки для компанії, які дає велосипеди в оренду, ви можете зробити з цього графіку? 3 основних висновки.

In [5]:
df['season'] = df['season'].astype('category')

fig = px.scatter(df,
                 x=['casual','registered'],
                 y='count',
                 size='count',
                 color='season',
                 hover_data=['datetime'],
                 trendline='ols',
                 title='Аналіз користувачів, що орендують велосипеди')
fig.show()

# 1.Це тренди для користувачів: чим більше зареєстрованих, тим більше к-сть оренди,
#   або ж пряма кореляція (що видно з trendline) та практично непомітна відсутність кореляції для випадкових користувачів.
# 2. -Заохочувати користувачів до реєстрації;
#    -У 2, 3 сезонах бути готовим до більшого попиту на оренду.
#    -У 1 та 4 сезонах активніше просувати маркетингові заходи для підвищення попиту.

## Завдання 3: Порівняння Plotly Express vs Graph Objects

**Завдання:**
Створіть лінійний графік помісячної динаміки оренди (кількість оренд) велосипедів двома способами - з Plotly Express та з Graph Objects.

**Дайте відповіді на питання.**
1. Як ви розумієте основну різницю між цими двома підходами?
2. Коли краще використовувати Plotly Express?
3. Коли потрібен Graph Objects?


In [6]:
monthly_count= df.resample('ME', on='datetime')['count'].sum().reset_index()

fig = px.line(
    monthly_count, x='datetime', y='count', title='Помісячна динаміка оренди велосипедів з Plotly Express')
fig.show()

In [21]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=monthly_count['datetime'],
        y=monthly_count['count'],
        mode='lines',
        hovertemplate='дата: %{x}<br>оренда: %{y}'
    ),
)
fig.update_layout(
    title='Помісячна динаміка оренди велосипедів з Graph Objects',
    xaxis_title='datetime',
    yaxis_title='count'
)
fig.show()

# 1-3. Як бачимо, для побудови аналогічних простих графіків, з  нам знадобилось
#   набагато більше коду з Graph Objects. Отже, при побудові типових простих
#   графіків краще використовувати метод Plotly Express, а от якщо знадобиться
#    більше кастомних налаштувань, або побудувати дашборд з кількома графіками -
#   обираємо Graph Objects.


## Завдання 4 (Опціональне): Дашборд з make_subplots (Graph Objects)

**Завдання:**
Створіть дашборд з 4 різними графіками в одній фігурі:
- Bar chart - середні значення загальної кількості оренд велосипедів за сезонами(кварталами)
- Pie chart - відсоткове співвідношення погодних умов в даних
- Line chart - середнє значення загальної кількості оренд велосипедів за годинами протягом доби
- Scatter plot - кореляція температури vs вологість

Додайте заголовок на дашборд.

**Дайте відповідь на питання**
- На ваш погляд, яка перевага об'єднання графіків в один дашборд?

In [110]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Cередня величина оренди за сезонами Індії',
                    'Співвідношення погодних умов',
                    'Cередня величина оренди погодинно',
                    'Температура vs вологість'),
    vertical_spacing=0.1,
    horizontal_spacing=0.1,
    specs=[[{"type": "xy"}, {"type": "domain"}],
           [{"type": "xy"}, {"type": "xy"}]])

#  візьмемо за основу індійські сезони
season_mean_count= df.groupby('weather_season_india')['count'].mean().round(0)
fig.add_trace(
    go.Bar(
        x=season_mean_count.index,
        y=season_mean_count,
        hovertemplate='Сезон: %{x}<br>Оренда: %{y}',
        showlegend=False
    ),
    row=1, col=1
)

grouped_weather = df['weather'].value_counts()
fig.add_trace(
    go.Pie(
        values=grouped_weather,
        labels=['1-ясно', '2-туман', '3-легкий дощ', '4-сильний дощ'],
        name='Погода',
        texttemplate='%{percent:.2%}'
    ),
      row=1, col=2
)

hourly_mean_count = df.groupby('hour')['count'].mean().round(0)
fig.add_trace(
    go.Scatter(mode='lines+markers',
               x=hourly_mean_count.index,
               y=hourly_mean_count,
               hovertemplate='Година: %{x}<br>Оренда: %{y}<br>',
               showlegend=False
               ),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(y=df['humidity'],
               x=df['temp'],
               mode='markers',
               hovertemplate='Вологість: %{x}<br>Температура: %{y}<br>',
               showlegend=False
    ),
    row=2, col=2
)

fig.update_layout(
    height=700,
    width=1000,
    title_text='Комплексний дашборд показників оренди велосипедів',
    title_font=dict(size=24),
    title_x=0.5
)

fig.show()

#  Як на мене, поєднання графіків набагато наочніше, дозволяє побачити всі важливі
#  інсайти без прокручувань, оскільки переглядати по окремості кожен графік не зручно
#  і не так ефектно при презентації.

## Завдання 5 (Опціональне): 3D візуалізація

**Завдання:**
Створіть 3D scatter plot для аналізу взаємозв'язку температури, швидкості вітру та загальної кількості орендованих велосипедів. Колір встановіть за сезоном(кварталом), а розмір - за загальною кількість оренд також.

Дайте відповіді на питання.
**Питання для інтерпретації:**
1. Яку додаткову інформацію, на ваш погляд, дає 3D візуалізація?
2. Чи видно кластери в 3D просторі?
3. Чи ви можете зробити висновки з цієї візуалізації, чи вам було простіше побудувати кілька 2D?



In [132]:
fig = px.scatter_3d(df.sample(1000).reset_index(),
                    x='temp',
                    y='windspeed',
                    z='count',
                    color='season',
                    size='count',
                    title='3D аналіз оренди велосипедів'
)
fig.update_layout(
    height=800,
    title_x=0.5
)
fig.show()

# 1. 3D скатерплот дозволяє  проаналізувати принаймі 5 факторів одночасно
#   (3 осі, колір та розмір) для кожної точки, або ж попарно ті фактори, які захочемо.
#   Це не просто для моментального сприйняття, але для досвідченого аналітика може
#   бути корисно.
# 2. Так
# 3. Кілька 2D ніяк не можуть замінити 3D, оскільки будуть певні викривлення
#   (різні масштаби шкал), та і це треба будувати багато графіків, що ускладнило б
#  інтерпритацію результатів, і нам неможливо було б проаналізувати 1 точку водночас
#  більше ніж на 2 показники.


## Завдання 6: Експорт та збереження інтерактивних графіків

**Завдання:**
Збережіть побудований попередній графік на plotly в формат HTML. Також змініть вручну щось на графіку (зум, виділення частини графіку) і збережіть його як статичне зображення через іконку фотоапарату у формат PNG. Завантажте файли з графіком у HTML та PNG (або посилання на них на github) разом з посиланням на цей ноутбук при здачі ДЗ.


In [133]:
fig.write_html('3D візуалізація оренди вел.')